# Projekt 5: Generacja animacji stickmana

Celem projektu było przygotowanie rozwiązania opartego o sieć dyfuzyjną, które generuje animację szkieletu człowieka na podstawie etykiety tekstowej ruchu. Model obsługuje dwa typy ruchu: `walk` oraz `jump`.

Wynikiem generowania jest sekwencja o kształcie `[48, 15, 3]`, czyli 48 klatek, 15 punktów kluczowych oraz 3 współrzędne przestrzenne dla każdego punktu.

## Zakres rozwiązania

W projekcie wykorzystano dane motion capture z datasetu CMU Mocap. Dane BVH zostały przetworzone do wspólnej reprezentacji szkieletu stickmana. Każda próbka została skrócona lub dopełniona do 48 klatek, a pozycje punktów zostały zapisane względem miednicy, aby ograniczyć wpływ globalnego przesunięcia postaci.

Model generuje animacje dla dwóch klas ruchu:

- `walk` - chód,
- `jump` - skok.

## Reprezentacja szkieletu

Szkielet składa się z 15 węzłów:

| Indeks | Punkt |
|---:|---|
| 0 | głowa |
| 1 | szyja |
| 2 | miednica |
| 3 | prawy bark |
| 4 | prawy łokieć |
| 5 | prawy nadgarstek |
| 6 | lewy bark |
| 7 | lewy łokieć |
| 8 | lewy nadgarstek |
| 9 | prawe biodro |
| 10 | prawe kolano |
| 11 | prawa kostka |
| 12 | lewe biodro |
| 13 | lewe kolano |
| 14 | lewa kostka |

Do wizualizacji punkty są łączone zgodnie z naturalną strukturą ciała: miednica-szyja-głowa, kończyny górne oraz kończyny dolne.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name != "5" and (PROJECT_DIR / "5").exists():
    PROJECT_DIR = PROJECT_DIR / "5"j

OUTPUT_DIR = PROJECT_DIR / "output"
RESULTS_PATH = OUTPUT_DIR / "evaluation_results.csv"

RESULTS_PATH

ModuleNotFoundError: No module named 'pandas'

## Architektura modelu

Zastosowano model `MotionTransformerDiffusion`, który przyjmuje zaszumioną sekwencję ruchu oraz warunek klasowy (`walk` albo `jump`). Sekwencja `[48, 15, 3]` jest spłaszczana w każdej klatce do wektora 45 cech, a następnie przetwarzana przez enkoder Transformer.

W modelu wykorzystano:

- projekcję wejścia do przestrzeni ukrytej,
- kodowanie pozycyjne dla kolejnych klatek,
- embedding kroku dyfuzji,
- embedding klasy ruchu,
- enkoder Transformer,
- projekcję wyjściową z powrotem do kształtu `[48, 15, 3]`.

Funkcja straty łączy kilka składników: błąd pozycji, prędkości, przyspieszenia, długości kości oraz stabilności stóp.

## Wygenerowane animacje

Poniżej znajdują się animacje wygenerowane przez model dla obu obsługiwanych rodzajów ruchu.

In [4]:
walk_gif = OUTPUT_DIR / "walk_generated.gif"
jump_gif = OUTPUT_DIR / "jump_generated.gif"

print("walk:", walk_gif)
display(Image(filename=str(walk_gif)))

print("jump:", jump_gif)
display(Image(filename=str(jump_gif)))

NameError: name 'OUTPUT_DIR' is not defined

## Metryki ewaluacyjne

Do oceny jakości wygenerowanych ruchów użyto trzech miar wymaganych w treści projektu:

- **FMD** (*Frechet Motion Distance*) - miara podobieństwa rozkładów ruchu rzeczywistego i wygenerowanego; niższa wartość oznacza większe podobieństwo.
- **MPJPE** (*Mean Per Joint Position Error*) - średni błąd położenia punktów szkieletu; niższa wartość oznacza dokładniejszą rekonstrukcję lub bardziej spójne pozycje stawów.
- **Var** - wariancja między wygenerowanymi próbkami; wyższa wartość oznacza większą różnorodność generacji.

In [ ]:
results = pd.read_csv(RESULTS_PATH)
results

Tabela wyników uzyskanych dla poszczególnych ruchów:

| Ruch | FMD | MPJPE | Var |
|---|---:|---:|---:|
| walk | 6.3021 | 2.3982 | 3.9003 |
| jump | 6.8159 | 2.4676 | 4.0938 |

In [ ]:
ax = results.set_index("Ruch")[["FMD", "MPJPE", "Var"]].plot(
    kind="bar",
    figsize=(8, 4),
    rot=0,
    title="Porównanie wyników ewaluacji dla walk i jump",
)
ax.set_xlabel("Ruch")
ax.set_ylabel("Wartość metryki")
ax.grid(axis="y", alpha=0.25)

## Interpretacja wyników

Dla ruchu `walk` uzyskano niższe wartości FMD oraz MPJPE niż dla ruchu `jump`, co sugeruje, że model lepiej odwzorował chód. Jest to zgodne z intuicją, ponieważ chód jest ruchem bardziej okresowym i zwykle łatwiejszym do nauczenia niż skok.

Ruch `jump` ma natomiast wyższą wariancję między wygenerowanymi próbkami. Oznacza to większą różnorodność generacji, ale jednocześnie może prowadzić do nieco większego błędu pozycji stawów.

Wyniki pokazują, że model potrafi generować animacje dla obu wymaganych klas ruchu, a wygenerowane próbki można zwizualizować jako animowany szkielet 3D.

## Podsumowanie

Zaimplementowane rozwiązanie:

- obsługuje generację dla dwóch poleceń tekstowych: `walk` i `jump`,
- generuje tensor animacji o kształcie `[48, 15, 3]`,
- wykorzystuje szkielet zawierający wymagane punkty kluczowe,
- zapisuje wizualizacje wygenerowanych animacji do plików GIF,
- raportuje metryki FMD, MPJPE oraz wariancję między próbkami.

Najlepsze wyniki jakościowe uzyskano dla ruchu `walk`, natomiast `jump` wykazał większą różnorodność generowanych próbek.